In [8]:
import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\agust\AppData\Local\Programs\Python\Python312\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\agust\AppData\Local\Programs\Python\Python312\python.exe"

print(os.environ["PYSPARK_PYTHON"])
print(os.environ["PYSPARK_DRIVER_PYTHON"])

C:\Users\agust\AppData\Local\Programs\Python\Python312\python.exe
C:\Users\agust\AppData\Local\Programs\Python\Python312\python.exe


In [9]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from pyspark import SparkContext
try:
    sc.stop()
except:
    pass

In [15]:
import os
from pyspark import SparkConf, SparkContext
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

conf = (SparkConf()
        .setMaster("local[*]")        # Corre Spark en tu máquina local, usando todos los núcleos disponibles del CPU ("local" a secas, corre Spark en modo local con un solo hilo (1 CPU))
        .setAppName("2.3.1 MapReduce con Spark")
        .set("spark.driver.bindAddress", "127.0.0.1")
        .set("spark.driver.host", "127.0.0.1"))
sc = SparkContext(conf = conf)
sc

<SparkContext master=local[*] appName=2.3.1 MapReduce con Spark>

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark") \
    .getOrCreate()


+------+----+
|nombre|edad|
+------+----+
|   Ana|  23|
|  Luis|  31|
| Pedro|  19|
+------+----+



### Procedimiento 1: ¿Cuál es el sueldo promedio de los médicos según su especialidad?

In [ ]:
# Cargamos los datos de médico:

df_medicos = spark.read.csv("medico",header= False)

# Los convertimos a dataset distribuido:

rdd_medicos = df_medicos.rdd 

In [ ]:
sorted((
    rdd_medicos
    # Fase de map: clave especialidad, valor (sueldo,1).
    .map(lambda medico: (medico[3],(float(medico[1]),1))) 
    # Fase de Shuffle y Reduce: se agrupa por clave y se suman los valores
    .reduceByKey(lambda tupla1, tupla2: (tupla1[0] + tupla2[0],tupla1[1] + tupla2[1]))
    # Calculamos el promedio y redondeamos
    .mapValues(lambda tupla: round(tupla[0]/tupla[1],2))

).collect(),key=lambda a:-a[1])
#El resultado es luego ordenado de forma decreciente según el sueldo promedio.

[('cardiologo', 11000.0), ('pediatra', 5000.0)]

### Procedimiento 2: ¿Cuáles son los diagnosticos más frecuente? ¿y los menos?

In [106]:
# Cargamos los datos de turno:

df_turno = spark.read.csv("turno",header= False)

# Los convertimos a dataset distribuido:

rdd_turnos = df_turno.rdd 

In [ ]:
res_diagnosticos = sorted((
    rdd_turnos
    # Fase de map: clave: diagnostico, valor: 1.
    .map(lambda turno: (turno[1],1)) 
    # Fase de Shuffle y Reduce: se agrupa por clave y se suman los valores para obtener el conteo
    .reduceByKey(lambda valor1, valor2: (valor1 + valor2))
).collect(),key=lambda a:-a[1])
#El resultado es luego ordenado de forma decreciente según el conteo.

In [ ]:
print("Diagnosticos más frecuentes:\n")
for i in range(2):
    print('Se diagnosticó', res_diagnosticos[i][0], res_diagnosticos[i][1],'veces')
print("\nDiagnosticos menos frecuentes:\n")
for i in range(2):
    print('Se diagnosticó', res_diagnosticos[-i][0], res_diagnosticos[-i][1],'veces')

Diagnosticos más frecuentes:

Se diagnosticó gripe 2 veces
Se diagnosticó faringitis 1 veces

Diagnosticos menos frecuentes:

Se diagnosticó gripe 2 veces
Se diagnosticó faringitis 1 veces


### Procedimiento 3: ¿Cuántos turnos tomó cada médico?

In [123]:
rdd_turnos.collect()

[Row(_c0='4', _c1='gripe', _c2='10000.00', _c3='2026-03-12', _c4='01:20:00', _c5='realizado', _c6='1234567893 ', _c7='1234567890 ', _c8='1'),
 Row(_c0='5', _c1='gripe', _c2='12000.00', _c3='2026-04-12', _c4='01:20:00', _c5='realizado', _c6='1234567893 ', _c7='1234567890 ', _c8='2'),
 Row(_c0='6', _c1='faringitis', _c2='14000.00', _c3='2026-05-12', _c4='01:20:00', _c5='realizado', _c6='1234567893 ', _c7='1234567891 ', _c8='3')]

In [128]:
# usando los mismos datos que la anterior consulta:
res_cant_turnos_por_med = sorted((
    rdd_turnos
    # Fase de map: clave: cuil_medico, valor: 1.
    .map(lambda turno: (int(turno[7]),1)) 
    # Fase de Shuffle y Reduce: se agrupa por clave y se suman los valores para obtener el conteo
    .reduceByKey(lambda valor1, valor2: (valor1 + valor2))
).collect(),key=lambda a:-a[1])
#El resultado es luego ordenado de forma decreciente según el conteo.

In [130]:
print("Cantidad de turnos por médico:")
for medico in res_cant_turnos_por_med:
    print(medico)


Cantidad de turnos por médico:
(1234567890, 2)
(1234567891, 1)
